# StockSense Pipeline Runner
**Run this notebook in Google Colab to populate the Neon database.**

----

## Before you run anything, read this doc first

### 1. Enable GPU (required for FinBERT in Cell 7)
- Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU** -> Save
- Do this BEFORE running any cells
- If you already started without GPU: Runtime -> Restart session, then re-run from the top

### 2. Set up Colab Secrets (required , do this once)
Click the key icon in the left sidebar, then add:

| Secret Name   | What to put there                                      |
|---------------|--------------------------------------------------------|
| `sshkey`     | Your GitHub SSH private key (`~/.ssh/id_ed25519`)      |
| `DATABASE_URL`| The Neon PostgreSQL connection string (get from Katie) |
| `HUGGINGFACE_HUB_TOKEN` | Kevin's HuggingFace API token (from hf.co/settings/tokens) |

Toggle Notebook access ON for all three secrets.

Your SSH key must have read access to the SCCapstone/ai_roosters repo.

Ask Katie for the DATABASE_URL if you don't have it.

### 3. Run order
| Cell | What it does | Required? |
|------|-------------|-----------|
| 1 | Clone repo | Yes |
| 2 | Navigate to backend | Yes |
| 3 | Install dependencies | Yes |
| 4 | Load secrets into env | Yes |
| 5 | Create DB tables | Yes (first time only) |
| 6 | Ingest stock price data | Yes |
| 6b | Ingest news articles from HuggingFace | Yes, must run before Cell 7 |
| 7 | Run FinBERT sentiment scoring | Only after Cell 6b |
| 8 | Run sentiment aggregator + XGBoost | Only after Cell 7 |

### 4. Article data (Cell 6b)
Cell 6b downloads financial news articles from the public HuggingFace dataset
`Brianferrell787/financial-news-multisource` and stores them in the `articles` table.
Cell 7 then scores those articles with FinBERT (positive / negative / neutral).

- Requires `HUGGINGFACE_HUB_TOKEN` in Colab Secrets
- Downloads ~2,500 articles (500 per year, 2020–2024)
- Safe to re-run. uses `ON CONFLICT DO NOTHING`

### 5. Never commit our secrets
Before saving this notebook to GitHub:
**Edit -> Clear all outputs** -> this removes any printed values from cells

In [ ]:
# ============================================================
# CELL 1: Clone private repo using SSH key from Colab Secrets
# ============================================================
import os
from google.colab import userdata

# Write private key to disk (fix Windows line endings)
os.makedirs("/root/.ssh", exist_ok=True)
key = userdata.get("sshkey")
key = key.replace("\r\n", "\n").replace("\r", "\n")
if not key.endswith("\n"):
    key += "\n"

with open("/root/.ssh/id_ed25519", "w") as f:
    f.write(key)
os.chmod("/root/.ssh/id_ed25519", 0o600)

# Add GitHub to known hosts (prevents interactive prompt)
!ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null

# Clone the repo
!git clone git@github.com:SCCapstone/ai_roosters.git

print("Clone complete.")


In [ ]:
# ============================================================
# CELL 2: Navigate into the backend folder
# ============================================================
%cd /content/ai_roosters/backend

# Verify in the right place
!pwd
!ls  # should show: app/, requirements-pipeline.txt, API.Dockerfile, etc.

In [ ]:
# Cell 3 - install system build tools first, then packages
!apt-get install -y --quiet libpq-dev gcc build-essential
#!pip install -r requirements-pipeline.txt --prefer-binary 2>&1 | tail -50
# Install core packages first (known-good binary wheels)
!pip install --prefer-binary \
    fastapi==0.95.2 uvicorn==0.22.0 sqlalchemy==1.4.41 \
    psycopg2-binary==2.9.6 "pydantic[email]==1.10.9" \
    python-jose[cryptography] "passlib[bcrypt]==1.7.4" bcrypt==4.0.1 \
    pandas numpy yfinance requests httpx python-dotenv \
    email-validator beautifulsoup4 tqdm \
    torch transformers xgboost scikit-learn datasets matplotlib

# Skip papermill install, it causes errors.
# Papermill is not needed for running the pipeline scripts

In [ ]:
# ============================================================
# CELL 4: Load secrets into environment variables
# DATABASE_URL is the Neon connection string - stored in
# Colab Secrets (ask Katie for URL)
# ============================================================
from google.colab import userdata
import os, sys

os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
os.environ["HUGGINGFACE_HUB_TOKEN"] = userdata.get("HUGGINGFACE_HUB_TOKEN")
os.environ["HF_TOKEN"] = os.environ["HUGGINGFACE_HUB_TOKEN"]

# add backend to Python path so imports work
sys.path.insert(0, '/content/ai_roosters/backend')

print("Environment ready.")
print(f"DATABASE_URL set: {'yes' if os.environ.get('DATABASE_URL') else 'NO - check the secret name'}")
print(f"HUGGINGFACE_HUB_TOKEN set: {'yes' if os.environ.get('HUGGINGFACE_HUB_TOKEN') else 'NO - check the secret name'}")

In [ ]:
# ============================================================
# CELL 5: Create all tables in Neon (safe to re-run ,
# uses CREATE TABLE IF NOT EXISTS )
# ============================================================

from app.db_init import init_db

init_db()
print("Database tables initialized.")



In [ ]:
# ============================================================
# CELL 6: Download price data from yfinance
# and store it in the Neon 'stocks' table.
#
# update_existing=False means: skip rows that already exist.
# Safe to re-run, won't create duplicates.
# ============================================================
from app.services.ingesting_pipelines.prices_ingest import PriceIngestor
from datetime import date

ingestor = PriceIngestor(os.environ["DATABASE_URL"])

tickers = [
    "AAPL",        # Apple
    "TSLA",        # Tesla
    "MSFT",        # Microsoft
    "GOOGL",       # Alphabet / Google
    "AMZN",        # Amazon
    "META",        # Meta (Facebook)
    "NVDA",        # Nvidia
    "JPM",         # JPMorgan Chase
    "BP",          # BP
    "RELIANCE.NS", # Reliance Industries
    "KSS",         # Kohl's
    "ALK",         # Alaska Air
    "NVS",         # Novartis
    "AXP",         # American Express
]

ingestor.ingest_multiple_stocks(
    tickers=tickers,
    start_date="2020-01-01",
    end_date=str(date.today()),   # fetch up to today
    period=None,
    update_existing=False,
)

print("Price ingestion complete.")

In [ ]:
# ============================================================
# CELL 6b: Download financial news articles from HuggingFace
# and store them in the Neon 'articles' table.
#
# MUST run before Cell 7 (FinBERT needs articles in the DB).
# Safe to re-run. uses ON CONFLICT DO NOTHING (no duplicates).
#
# Downloads ~2,500 articles (500 per year, 2020–2024).
# Takes roughly 5–20 minutes depending on dataset stream speed.
#
# NOTE: fallback_to_non_streaming_after is set very high and
# fallback_if_target_year_share_below=0.0 to prevent the
# code from switching to non-streaming mode, which would try
# to download the entire 57M-row dataset and fill Colab disk.
# ============================================================
from app.services.ingesting_pipelines.news_ingest import ArticleIngestor

print("Starting article ingestion from HuggingFace...")
print("This may take 5–20 minutes. Watch the progress logs below.")

ArticleIngestor(os.environ["DATABASE_URL"]).ingest_all_years_one_pass(
    years=[2020, 2021, 2022, 2023, 2024],
    per_year=500,           # 500 articles per year = 2,500 total
    end_date="2024-07-30",  # matches FinBERT default window
    max_scanned=10_000_000, # scan up to 10M rows (stays in streaming)
    flush_batch_size=2000,
    streaming=True,
    fallback_to_non_streaming_after=100_000_000,  # effectively disabled
    fallback_if_target_year_share_below=0.0,       # effectively disabled
)

print("Article ingestion complete.")

In [ ]:
# ============================================================
# CELL 7: Score articles with FinBERT and store results in
# the Neon 'articles' table.
#
# Reads articles directly from the DB (no CSV needed).
# Must run Cell 6b before this cell.
# Requires T4 GPU runtime. Will be very slow on CPU.
#
# Scores each article as positive / negative / neutral and
# writes sentiment_score, prob_pos, prob_neg, prob_neu back.
# Safe to re-run, only processes articles missing sentiment.
# ============================================================
from app.services.sentiment.article_processing import (
    get_articles_table,
    fetch_articles_from_db,
    finbert_score,
    write_sentiment_to_db,
    FetchFromDBArtifact,
    DEFAULT_START_DATE,
    DEFAULT_END_DATE,
    DEFAULT_TOTAL_TO_PROCESS,
    DEFAULT_FETCH_BATCH,
)
from sqlalchemy import create_engine

db_url = os.environ["DATABASE_URL"]
engine = create_engine(db_url)
articles_tbl, col_list = get_articles_table(engine)
cols = set(col_list)

start_date   = os.getenv("FINBERT_START_DATE", "").strip() or DEFAULT_START_DATE
end_date     = os.getenv("FINBERT_END_DATE",   "").strip() or DEFAULT_END_DATE
total_target = int(os.getenv("FINBERT_TOTAL", str(DEFAULT_TOTAL_TO_PROCESS)))
fetch_batch  = int(os.getenv("FINBERT_FETCH_BATCH", str(DEFAULT_FETCH_BATCH)))

print(f"FinBERT run: window={start_date} -> {end_date}, target={total_target} articles")

total_fetched = 0
total_written = 0
loop = 0

while total_fetched < total_target:
    loop += 1
    remaining  = total_target - total_fetched
    this_limit = min(fetch_batch, remaining)

    fetch_art = FetchFromDBArtifact(
        limit=this_limit,
        start_date=start_date,
        end_date=end_date,
        only_missing_sentiment=True,
    )

    batch = fetch_articles_from_db(engine, articles_tbl, cols, fetch_art)
    n = len(batch.url)

    if n == 0:
        print("No more unscored articles found. Stopping.")
        break

    print(f"[Loop {loop}] Scoring {n} articles with FinBERT...")
    scored  = finbert_score(batch)
    written = write_sentiment_to_db(engine, articles_tbl, cols, scored)

    total_fetched += n
    total_written += written
    print(f"[Loop {loop}] Wrote {written} rows. Total so far: {total_written}")

    if n < this_limit:
        print("Fetched fewer than requested; likely exhausted. Stopping.")
        break

print(f"\nFinBERT complete: {total_written} articles scored in {loop} loop(s).")

In [ ]:
# ============================================================
# CELL 8: Join articles + stocks to produce per-ticker
# daily sentiment summaries in 'sentiment_snapshots' table.
# Run AFTER Cell 6 (prices) AND Cell 7 (articles)complete.
# ============================================================
from app.services.sentiment.aggregator import run_sentiment_snapshot_pipeline_from_env

run_sentiment_snapshot_pipeline_from_env()
print("Sentiment aggregation complete.")
